# PYTORCH MASTERY — NOTEBOOK 13
## Chapters 72–78: Model Checkpointing → Custom Training Architectures → Custom Autograd → PyTorch 2.x Compilation → Performance Profiling → Memory Optimization → Gradient Clipping

Continuing directly from Notebook 12.


### PYTORCH MASTERY PROGRESS
Current Chapter: 78 | Current Level: Advanced Engineering

- [x] Chapters 1–71 *(Notebooks 1–12)*
- [x] Model Checkpointing
- [x] Custom Training Architectures
- [x] Custom Autograd
- [x] PyTorch 2.x Compilation
- [x] Performance Profiling
- [x] Memory Optimization
- [x] Gradient Clipping


---
# CHAPTER 72 — MODEL CHECKPOINTING

## 1. Concept
A **checkpoint** saves not just a model's weights, but everything needed to **resume training exactly where it left off** — model state, optimizer state (momentum/Adam moments), scheduler state, epoch number, and any other training metadata — directly extending the Scikit-learn course's Chapter 67 (Model Versioning) with the specific additional requirement of mid-training resumability.

## 2. Why Optimizer State Must Be Saved Too


In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(10, 5)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("""
Recall Chapter 20: Adam/AdamW maintain PER-PARAMETER running estimates of the
gradient's mean and squared magnitude (the first and second "moments").
If you save and later reload ONLY the model's weights (state_dict()), then
create a FRESH optimizer, that optimizer starts with ZERO accumulated
momentum/moment history -- training resumes with WORSE optimization dynamics
than if training had genuinely continued uninterrupted, even though the
MODEL's weights themselves are identical.

Saving optimizer.state_dict() ALONGSIDE model.state_dict() and restoring BOTH
is what allows training to resume with IDENTICAL dynamics to an uninterrupted
run -- genuinely important for long training runs that may need to be paused
and resumed (e.g., due to a cluster maintenance window, a preemptible cloud
instance being reclaimed, or simply wanting to extend training after initially
stopping).
""")



Recall Chapter 20: Adam/AdamW maintain PER-PARAMETER running estimates of the
gradient's mean and squared magnitude (the first and second "moments").
If you save and later reload ONLY the model's weights (state_dict()), then
create a FRESH optimizer, that optimizer starts with ZERO accumulated
momentum/moment history -- training resumes with WORSE optimization dynamics
than if training had genuinely continued uninterrupted, even though the
MODEL's weights themselves are identical.

Saving optimizer.state_dict() ALONGSIDE model.state_dict() and restoring BOTH
is what allows training to resume with IDENTICAL dynamics to an uninterrupted
run -- genuinely important for long training runs that may need to be paused
and resumed (e.g., due to a cluster maintenance window, a preemptible cloud
instance being reclaimed, or simply wanting to extend training after initially
stopping).



## 3. A Complete, Production-Quality Checkpoint Function


In [ ]:
import os
import torch
import torch.nn as nn

# Define model, optimizer, scheduler so the code can execute correctly
model = nn.Linear(10, 5)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

def save_checkpoint(model, optimizer, scheduler, epoch, best_val_loss, path):
    """Saves EVERYTHING needed to resume training exactly -- directly extending
    the Scikit-learn course's Ch 67 versioning discipline with training-resumability."""
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "best_val_loss": best_val_loss,
        "pytorch_version": torch.__version__,   # Chapter 66's (Scikit-learn course) version-pinning lesson
    }
    torch.save(checkpoint, path)


def load_checkpoint(model, optimizer, scheduler, path, device):
    # Fix: PyTorch 2.6 default is weights_only=True. Since we are saving torch.__version__ (a TorchVersion object),
    # we must explicitly set weights_only=False (if we trust the source) or add TorchVersion to safe globals.
    checkpoint = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scheduler is not None and checkpoint["scheduler_state_dict"] is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    return checkpoint["epoch"], checkpoint["best_val_loss"]


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
os.makedirs("/home/claude/checkpoints", exist_ok=True)
save_checkpoint(model, optimizer, scheduler, epoch=5, best_val_loss=0.342, path="/home/claude/checkpoints/ckpt_epoch5.pt")
print("Checkpoint saved.")

# Simulate RESUMING on a fresh model/optimizer/scheduler (e.g., in a new process)
fresh_model = nn.Linear(10, 5)
fresh_optimizer = torch.optim.AdamW(fresh_model.parameters(), lr=1e-3)
fresh_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(fresh_optimizer, T_max=20)

resumed_epoch, resumed_best_loss = load_checkpoint(
    fresh_model, fresh_optimizer, fresh_scheduler, "/home/claude/checkpoints/ckpt_epoch5.pt",
    device=torch.device("cpu"),
)
print(f"Resumed from epoch {resumed_epoch}, best_val_loss={resumed_best_loss}")


Checkpoint saved.
Resumed from epoch 5, best_val_loss=0.342


### Line-by-Line Explanation
- `map_location=device` in `torch.load()` — **essential**, not optional, for any checkpoint that might be loaded on different hardware than it was saved on: a checkpoint saved from a GPU-resident model contains tensors tagged with that GPU device; loading it directly on a machine without a GPU (or with a different GPU configuration) would raise an error without `map_location` explicitly remapping those tensors to the loading machine's available device — directly Chapter 27's device-placement discipline, now applied to checkpoint loading specifically.
- Saving `pytorch_version` inside the checkpoint itself — directly the Scikit-learn course's Chapter 66 environment-compatibility lesson, now embedded in the artifact itself for easy later inspection, rather than relying on separate, easily-lost documentation.
- Checkpointing the **scheduler's** state too — a scheduler like `CosineAnnealingLR` tracks its own internal step count to know where in its schedule it currently is; without restoring this, a resumed training run's learning rate schedule would restart from the beginning, rather than continuing correctly from where it left off.

### Common Mistakes
- Saving only `model.state_dict()` and assuming that's sufficient to resume training — as explained above, this loses optimizer momentum/moment history, genuinely affecting training dynamics upon resumption even though the model weights themselves are preserved correctly.
- Forgetting `map_location` when loading a checkpoint on different hardware than it was saved on — a common, easily-avoided `RuntimeError`.
- Not saving enough metadata (epoch number, best validation metric so far) to make an informed decision about how to actually resume (e.g., for correctly continuing early-stopping logic, Scikit-learn course Chapter 31).

### Interview Insight
**Q: Why isn't saving only `model.state_dict()` sufficient if you intend to later resume training (not just perform inference)?**
A: Adaptive optimizers like Adam/AdamW maintain per-parameter running statistics (first and second moment estimates) that are built up over many training steps and directly influence how future parameter updates are scaled — if you resume training with a freshly-initialized optimizer (zero accumulated history) even while loading the correct model weights, the optimizer's behavior for the next several steps will differ meaningfully from what it would have been had training genuinely continued uninterrupted, since it's effectively restarting its own internal adaptive learning-rate estimation from scratch. Saving and restoring `optimizer.state_dict()` alongside the model weights preserves this accumulated optimization state, allowing training to resume with dynamics matching an uninterrupted run — essential for genuinely correct pause/resume training workflows, not just for inference-only model deployment (where only the weights matter).

### Quiz (Chapter 72)
1. What does a full checkpoint need to contain beyond just model weights, and why each piece?
2. What does `map_location` solve, and why is it essential for cross-device checkpoint loading?
3. Why does the scheduler's state need to be saved, not just the model and optimizer?


---
# CHAPTER 73 — CUSTOM TRAINING ARCHITECTURES

## 1. Concept
Real-world problems often need architectures beyond a single-input, single-output sequential model — multiple input modalities, multiple prediction targets, or custom loss combinations. This chapter builds a genuinely multi-input, multi-output model, directly extending Chapter 16's `nn.Module` fundamentals.

## 2. A Multi-Input, Multi-Output Architecture


In [ ]:
import torch
import torch.nn as nn

class MultiTaskModel(nn.Module):
    """THREE input branches (e.g., tabular features, an image, and text) are
    processed SEPARATELY, then FUSED, then split into TWO separate prediction
    HEADS (e.g., a classification task AND a regression task, trained
    JOINTLY) -- a genuinely common real-world pattern."""

    def __init__(self, tabular_dim, image_feature_dim, text_feature_dim, fusion_dim=128):
        super().__init__()
        # Branch A: tabular features
        self.tabular_branch = nn.Sequential(nn.Linear(tabular_dim, 64), nn.ReLU())
        # Branch B: pre-extracted image features (e.g., from a CNN backbone, Chapter 38)
        self.image_branch = nn.Sequential(nn.Linear(image_feature_dim, 64), nn.ReLU())
        # Branch C: pre-extracted text features (e.g., from a Transformer encoder, Chapter 51)
        self.text_branch = nn.Sequential(nn.Linear(text_feature_dim, 64), nn.ReLU())

        # FUSION: combine all three branches' outputs
        self.fusion = nn.Sequential(nn.Linear(64 * 3, fusion_dim), nn.ReLU(), nn.Dropout(0.3))

        # TWO separate heads, sharing the fused representation
        self.classification_head = nn.Linear(fusion_dim, 5)   # e.g., 5-class classification
        self.regression_head = nn.Linear(fusion_dim, 1)         # e.g., a continuous value prediction

    def forward(self, tabular_x, image_x, text_x):
        tab_out = self.tabular_branch(tabular_x)
        img_out = self.image_branch(image_x)
        txt_out = self.text_branch(text_x)

        fused = torch.cat([tab_out, img_out, txt_out], dim=1)   # Chapter 11's cat -- CONCATENATE along features
        fused = self.fusion(fused)

        classification_logits = self.classification_head(fused)
        regression_output = self.regression_head(fused)

        return classification_logits, regression_output   # a TUPLE of TWO outputs


model = MultiTaskModel(tabular_dim=20, image_feature_dim=512, text_feature_dim=256)

batch_size = 8
tabular_input = torch.randn(batch_size, 20)
image_input = torch.randn(batch_size, 512)
text_input = torch.randn(batch_size, 256)

class_logits, reg_output = model(tabular_input, image_input, text_input)
print("Classification logits shape:", class_logits.shape)
print("Regression output shape:", reg_output.shape)


Classification logits shape: torch.Size([8, 5])
Regression output shape: torch.Size([8, 1])


### Line-by-Line Explanation — Combining Two Losses for Joint Training


In [ ]:
classification_criterion = nn.CrossEntropyLoss()
regression_criterion = nn.MSELoss()

class_targets = torch.randint(0, 5, (batch_size,))
reg_targets = torch.randn(batch_size, 1)

class_loss = classification_criterion(class_logits, class_targets)
reg_loss = regression_criterion(reg_output, reg_targets)

# COMBINING two losses -- a WEIGHTED sum, since the two losses may have very
# DIFFERENT natural scales/magnitudes (CrossEntropy and MSE are not directly comparable)
task_weight_classification = 1.0
task_weight_regression = 0.5   # a TUNABLE hyperparameter, balancing the two tasks' relative importance

combined_loss = (task_weight_classification * class_loss) + (task_weight_regression * reg_loss)
combined_loss.backward()   # ONE backward() call -- gradients flow back through BOTH heads AND the SHARED fusion/branches

print(f"Classification loss: {class_loss.item():.4f}")
print(f"Regression loss: {reg_loss.item():.4f}")
print(f"Combined (weighted) loss: {combined_loss.item():.4f}")


Classification loss: 1.6498
Regression loss: 0.3784
Combined (weighted) loss: 1.8390


### Why Loss Weighting Matters — the Scale Mismatch Problem
`CrossEntropyLoss` and `MSELoss` have fundamentally different natural scales and dynamics — without deliberate weighting, whichever task's loss happens to have a larger natural magnitude would dominate the combined gradient, effectively starving the other task of meaningful training signal, even if both tasks are equally important to the actual application. The weighting coefficients (`task_weight_classification`, `task_weight_regression`) are genuine hyperparameters requiring tuning — sometimes via systematic search (Scikit-learn course's hyperparameter tuning chapters), sometimes via more sophisticated adaptive weighting schemes (uncertainty-based task weighting, beyond this course's current scope) — to achieve a training balance that serves the application's actual priorities.

### Common Mistakes
- Combining multiple losses with equal (implicit `1.0`) weights without considering their natural scale differences — can silently cause one task to dominate training at the expense of the other.
- Forgetting that a single `combined_loss.backward()` call correctly backpropagates through **both** heads and all **shared** branches simultaneously — some engineers mistakenly believe they need separate `backward()` calls per task, which would be both unnecessary and (without careful `retain_graph` handling, Chapter 13) likely to raise an error on the second call.
- Not considering whether tasks might actually benefit from some **task-specific** branch capacity rather than being forced to share too much of the network — the fusion architecture's design (how much is shared vs. task-specific) is itself a genuine design choice worth deliberate consideration, not a default to apply unthinkingly.

### Interview Insight
**Q: You're training a model jointly on a classification task and a regression task using a combined, weighted loss. Why is loss weighting necessary, and how would you go about choosing appropriate weights?**
A: Different loss functions have fundamentally different natural scales and magnitudes — `CrossEntropyLoss` and `MSELoss`, for instance, aren't directly comparable in magnitude, and combining them with equal implicit weight risks letting whichever loss happens to be naturally larger dominate the combined gradient, effectively starving the other task's learning signal regardless of that task's actual importance to the application. Choosing appropriate weights typically starts with examining each loss's typical magnitude early in training to get a rough sense of the scale mismatch, then treating the weighting coefficients as genuine hyperparameters to tune — via systematic search (grid/random search over weight combinations, evaluating each task's own validation metric) or, in more sophisticated setups, adaptive schemes that dynamically balance task weights based on each task's uncertainty or training progress during training itself.

### Quiz (Chapter 73)
1. Why does a multi-input model process each input modality through a separate branch before fusing them?
2. Why is naive equal-weighting of two different loss functions potentially problematic?
3. Does combining two losses into one weighted sum before `.backward()` correctly update both task-specific heads AND shared layers? Why?


---
# CHAPTER 74 — CUSTOM AUTOGRAD

## 1. Concept
For the overwhelming majority of models, PyTorch's automatic differentiation (Chapter 13) handles gradient computation entirely transparently — but occasionally, you need to define a **custom operation** with a hand-specified forward and backward computation, via `torch.autograd.Function` — genuinely useful for implementing a novel mathematical operation, a numerically-stabilized version of an existing operation, or wrapping a non-differentiable external computation with an approximate gradient.

## 2. A Custom Autograd Function — Implementation


In [ ]:
import torch

class CustomReLU(torch.autograd.Function):
    """A from-scratch reimplementation of ReLU's forward AND backward computation --
    purely EDUCATIONAL here (nn.ReLU already exists and should be used in practice),
    but demonstrates EXACTLY how autograd's internals actually work."""

    @staticmethod
    def forward(ctx, input_tensor):
        """ctx (context) is used to SAVE information needed later, during backward()."""
        ctx.save_for_backward(input_tensor)   # SAVE the input -- needed to compute the gradient later
        return input_tensor.clamp(min=0)        # ReLU's forward computation: max(0, x)

    @staticmethod
    def backward(ctx, grad_output):
        """grad_output is the GRADIENT FLOWING IN from the NEXT operation in the
        computational graph (the chain rule's 'upstream' gradient, Chapter 14).
        This function must return the gradient with respect to EVERY INPUT
        forward() received (here, just one: input_tensor)."""
        input_tensor, = ctx.saved_tensors   # RETRIEVE what was saved during forward()

        grad_input = grad_output.clone()
        grad_input[input_tensor < 0] = 0   # ReLU's local derivative: 1 where input>0, EXACTLY 0 where input<0
                                              # (Chapter 18's "dead ReLU" discussion, now seen at the IMPLEMENTATION level)
        return grad_input


# Using the custom function -- .apply() is how torch.autograd.Function subclasses are invoked
x = torch.tensor([-2.0, -1.0, 0.5, 3.0], requires_grad=True)
y = CustomReLU.apply(x)
y.sum().backward()

print("Input:", x.detach())
print("Output (custom ReLU forward):", y.detach())
print("Gradient (custom ReLU backward):", x.grad)

# Verify it matches PyTorch's BUILT-IN ReLU exactly
x2 = torch.tensor([-2.0, -1.0, 0.5, 3.0], requires_grad=True)
y2 = torch.relu(x2)
y2.sum().backward()
print("\nBuilt-in ReLU gradient (for comparison):", x2.grad)
print("Match:", torch.equal(x.grad, x2.grad))


Input: tensor([-2.0000, -1.0000,  0.5000,  3.0000])
Output (custom ReLU forward): tensor([0.0000, 0.0000, 0.5000, 3.0000])
Gradient (custom ReLU backward): tensor([0., 0., 1., 1.])

Built-in ReLU gradient (for comparison): tensor([0., 0., 1., 1.])
Match: True


### Line-by-Line Explanation — What This Reveals About Autograd's Internals
- `ctx.save_for_backward(input_tensor)` / `ctx.saved_tensors` — this is the **actual mechanism** underlying every built-in PyTorch operation's `.grad_fn` (Chapter 13): the forward pass saves exactly the information its own backward computation will need, and the backward pass retrieves it — directly demystifying what "PyTorch automatically tracks the computational graph" concretely means at the implementation level.
- `backward(ctx, grad_output)`'s **signature** reveals the chain rule's actual mechanics precisely: `grad_output` is the gradient flowing in from whatever operation came *after* this one in the forward pass (the "upstream" gradient); this custom backward function's job is to compute the **local** derivative (here, ReLU's `1` or `0`) and **multiply** it by `grad_output` — exactly the chain rule's "multiply the local derivative by the upstream gradient" rule from Chapter 14, now made completely explicit in code rather than happening invisibly inside a built-in operation.
- The exact match with `torch.relu`'s gradient confirms this custom implementation is genuinely, numerically correct — not merely plausible-looking.

### Common Mistakes
- Forgetting `ctx.save_for_backward()` for any tensor the backward pass will need — causes an error or (worse) silently incorrect gradients if something else happens to be accessible instead.
- Returning the wrong number of gradients from `backward()` — must return exactly one gradient tensor (or `None`, for a non-differentiable input) per argument `forward()` received, in the same order.
- Writing a custom `Function` when a combination of existing, built-in differentiable operations would have worked just as well — custom autograd functions are a genuinely advanced, specialized tool; reach for them only when a standard composition of existing operations truly cannot express what's needed (e.g., for operations with no natural PyTorch-native differentiable implementation, or for a specific numerical-stability improvement over the default autograd-derived gradient).

### Interview Insight
**Q: When would you need to write a custom `torch.autograd.Function` rather than simply composing existing PyTorch operations?**
A: Custom autograd functions are genuinely needed in a few specific situations: implementing a mathematical operation that has no natural expression as a composition of PyTorch's existing differentiable operations; wrapping an external, non-differentiable computation (e.g., calling into a non-PyTorch library or an algorithm with no natural gradient) with a hand-specified, approximate or exact gradient so it can still participate in an end-to-end differentiable pipeline; or providing a numerically more stable backward computation than what PyTorch's automatic differentiation would derive by default from a straightforward forward implementation (certain operations can have numerically fragile "naive" gradients that a carefully hand-derived alternative avoids). For the large majority of standard model-building, composing existing `nn` layers and differentiable tensor operations is both sufficient and strongly preferred — custom autograd functions are a specialized, advanced tool reserved for genuine edge cases.

### Quiz (Chapter 74)
1. What does `ctx.save_for_backward()` actually accomplish, and why is it needed?
2. What does `grad_output` represent in a custom `backward()` method, and how does it relate to the chain rule?
3. Name two genuine, specific scenarios where a custom `torch.autograd.Function` is actually needed.


---
# CHAPTER 75 — PYTORCH 2.X COMPILATION

## 1. Concept
`torch.compile` (introduced in PyTorch 2.0) **compiles** a model's computation into an optimized, fused form before execution — directly addressing the historical performance gap between PyTorch's flexible "eager" execution (Chapter 1's define-by-run model) and the raw execution speed a static, compiled graph can achieve, **without** sacrificing eager mode's debugging/flexibility advantages during development.

## 2. Eager Execution vs. Compiled Execution


In [ ]:
import torch
import torch.nn as nn
import time

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        return self.net(x)


model = SimpleModel()
compiled_model = torch.compile(model)   # ONE LINE -- this is the entire API surface for the common case

x = torch.randn(64, 512)

print("""
torch.compile(model) returns a NEW callable that behaves IDENTICALLY to the
original model (same forward() interface, same output) but is INTERNALLY
optimized -- PyTorch analyzes the model's computation graph (using a component
called TorchDynamo to capture the graph, then further compiler backends to
optimize it) and FUSES multiple separate operations into fewer, more efficient
combined kernels, reducing the overhead of many small, separate GPU kernel
launches.

IMPORTANT: the FIRST call to a compiled model is SLOWER than eager execution --
this is when the ACTUAL compilation happens (a genuine, one-time cost).
SUBSEQUENT calls with the SAME input shapes/types reuse the compiled version
and can be MEANINGFULLY faster than eager execution, especially for larger,
more complex models with many small operations to fuse.
""")



torch.compile(model) returns a NEW callable that behaves IDENTICALLY to the
original model (same forward() interface, same output) but is INTERNALLY
optimized -- PyTorch analyzes the model's computation graph (using a component
called TorchDynamo to capture the graph, then further compiler backends to
optimize it) and FUSES multiple separate operations into fewer, more efficient
combined kernels, reducing the overhead of many small, separate GPU kernel
launches.

IMPORTANT: the FIRST call to a compiled model is SLOWER than eager execution --
this is when the ACTUAL compilation happens (a genuine, one-time cost).
SUBSEQUENT calls with the SAME input shapes/types reuse the compiled version
and can be MEANINGFULLY faster than eager execution, especially for larger,
more complex models with many small operations to fuse.



### Line-by-Line Explanation — Why the First Call Is Slower
- The first call triggers **tracing/compilation**: PyTorch's TorchDynamo component analyzes the model's actual Python bytecode execution to capture its computational graph, then compiler backends (commonly built on Triton for GPU kernels) generate optimized, fused kernel code for that specific graph — this analysis-and-compilation process takes real, measurable time.
- Every **subsequent** call with the same input shapes reuses this already-compiled artifact — this is exactly why `torch.compile` provides a genuine speedup specifically for training loops or repeated-inference scenarios (many calls with consistent shapes), but provides little to no benefit (and a real one-time cost) for a model called only once or twice.
- **Changing input shapes** between calls can trigger **recompilation** (PyTorch tries to be smart about reusing compiled artifacts across similar shapes where possible, but this isn't always guaranteed) — a genuine practical consideration for models that might see highly variable input shapes (e.g., variable sequence lengths in NLP without consistent padding).

## 3. When `torch.compile` Helps Most — and When It Doesn't


In [ ]:
print("""
torch.compile GENERALLY HELPS MOST for:
  - Models with MANY small operations (benefits substantially from kernel
    FUSION -- combining several small GPU operations into fewer, larger ones)
  - TRAINING loops or repeated inference with CONSISTENT input shapes
    (amortizing the one-time compilation cost across many subsequent calls)
  - Models NOT already dominated by a few very large, already-optimized
    operations (e.g., models ALREADY mostly spending time inside highly
    optimized cuDNN convolution/attention kernels may see LESS relative benefit)

torch.compile PROVIDES LESS BENEFIT (or a NET LOSS) for:
  - A model called only ONCE or a FEW times (compilation overhead isn't
    amortized across enough subsequent calls to pay for itself)
  - Models with HIGHLY VARIABLE input shapes (triggering frequent, costly
    recompilation)
  - Models with EXTENSIVE Python-level data-dependent control flow that
    TorchDynamo cannot cleanly trace/capture (falls back to eager execution
    for the un-traceable parts, reducing the overall benefit)

ALWAYS MEASURE the actual speedup on YOUR specific model/workload before
assuming torch.compile provides a benefit -- exactly the Scikit-learn course's
Chapter 61 "profile before optimizing" discipline, directly applicable here.
""")



torch.compile GENERALLY HELPS MOST for:
  - Models with MANY small operations (benefits substantially from kernel
    FUSION -- combining several small GPU operations into fewer, larger ones)
  - TRAINING loops or repeated inference with CONSISTENT input shapes
    (amortizing the one-time compilation cost across many subsequent calls)
  - Models NOT already dominated by a few very large, already-optimized
    operations (e.g., models ALREADY mostly spending time inside highly
    optimized cuDNN convolution/attention kernels may see LESS relative benefit)

torch.compile PROVIDES LESS BENEFIT (or a NET LOSS) for:
  - A model called only ONCE or a FEW times (compilation overhead isn't
    amortized across enough subsequent calls to pay for itself)
  - Models with HIGHLY VARIABLE input shapes (triggering frequent, costly
    recompilation)
  - Models with EXTENSIVE Python-level data-dependent control flow that
    TorchDynamo cannot cleanly trace/capture (falls back to eager execution
 

### Common Mistakes
- Benchmarking `torch.compile`'s speedup by timing only the **first** call — this measures compilation overhead, not the actual steady-state speedup; always exclude the first call (or a few "warmup" calls) from timing measurements.
- Expecting `torch.compile` to provide a dramatic benefit for every model unconditionally — the actual speedup varies substantially by model architecture and workload, and measuring on your specific case (not assuming from general claims) is essential.
- Using `torch.compile` with highly variable input shapes without considering the recompilation cost this can trigger — potentially negating the intended benefit entirely.

### Interview Insight
**Q: You apply `torch.compile` to a model and time a single forward pass, finding it's actually slower than eager execution. Does this mean `torch.compile` doesn't help for this model?**
A: Not necessarily — the very first call to a compiled model includes the one-time compilation cost itself (TorchDynamo tracing the computation graph and compiler backends generating optimized fused kernels), which is a genuine, measurable overhead unrelated to the compiled model's actual steady-state execution speed. The correct way to evaluate `torch.compile`'s benefit is to run several "warmup" calls first (triggering and absorbing the compilation cost), then measure the timing of subsequent calls with consistent input shapes — comparing that steady-state compiled performance against eager execution's steady-state performance is the fair, meaningful comparison; a single-call timing comparison will always unfairly penalize the compiled version.

### Quiz (Chapter 75)
1. Why is the first call to a `torch.compile`d model slower than eager execution?
2. What kind of model/workload benefits most from `torch.compile`, and what kind benefits least?
3. How should you correctly benchmark `torch.compile`'s actual speedup, avoiding a common measurement mistake?


---
# CHAPTER 76 — PERFORMANCE PROFILING

## 1. Concept
Beyond the systematic bottleneck checklist (Chapter 28), PyTorch's own built-in **profiler** gives detailed, fine-grained timing and memory breakdowns per operation — the precise tool for answering "why is my model taking 30 minutes when a comparable implementation takes 5."

## 2. `torch.profiler` — Detailed Operation-Level Timing


In [ ]:
import torch
import torch.nn as nn
from torch.profiler import profile, record_function, ProfilerActivity

model = nn.Sequential(
    nn.Linear(512, 512), nn.ReLU(),
    nn.Linear(512, 512), nn.ReLU(),
    nn.Linear(512, 10),
)
x = torch.randn(32, 512)

activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities.append(ProfilerActivity.CUDA)

with profile(activities=activities, record_shapes=True) as prof:
    with record_function("model_forward"):   # a NAMED region, for grouping/labeling specific code sections
        output = model(x)
    with record_function("loss_and_backward"):
        loss = output.sum()
        loss.backward()

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))
print("""
This table breaks down EXACTLY how much time each individual OPERATION
(each specific Linear layer's matmul, each ReLU, the backward pass's
corresponding gradient operations) consumed -- genuinely precise, actionable
information for finding the REAL bottleneck, rather than guessing.
record_function()'s NAMED regions let you group related operations together
in the report, making a complex model's profile much easier to read and reason about.
""")


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                      loss_and_backward        32.19%       3.528ms        57.08%       6.256ms       6.256ms             1  
                                          model_forward         4.60%     503.861us        42.92%       4.705ms       4.705ms             1  
                                           aten::linear        23.51%       2.577ms        37.87%       4.151ms       1.384ms             3  
    autograd::engine::evaluate_function: AddmmBackward0         0.49%      53.191us        21.27%       2.332ms     777.321us             3  
      

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


### Line-by-Line Explanation
- `ProfilerActivity.CPU`/`ProfilerActivity.CUDA` — specifies which hardware's operations to profile; including `CUDA` (when available) is essential for diagnosing GPU-bound bottlenecks specifically, since CPU-only profiling would miss GPU kernel execution time entirely.
- `record_shapes=True` — additionally records each operation's input tensor shapes in the profiling output — genuinely useful for spotting an unexpectedly large or small tensor flowing through a specific operation, which can itself be a clue toward a shape-related inefficiency or bug.
- `record_function("model_forward")` — a named marker letting you label specific sections of code in the profiling output (e.g., distinguishing "forward pass time" from "backward pass time" from "data loading time" at a glance), rather than having to infer this from a long, undifferentiated list of individual low-level operations.

## 3. Exporting a Profile for Visual Inspection


In [ ]:
print("""
prof.export_chrome_trace("trace.json")

The exported trace can be opened directly in Chrome's own built-in trace
viewer (chrome://tracing) or in specialized tools -- providing a VISUAL,
TIMELINE-based view of exactly what was happening on CPU and GPU at every
moment during the profiled code, including GAPS (idle time) that directly
reveal exactly WHERE and for HOW LONG the GPU was waiting (directly
Chapter 28's "GPU utilization" diagnostic, now visualized precisely rather
than inferred from an aggregate nvidia-smi percentage).
""")



prof.export_chrome_trace("trace.json")

The exported trace can be opened directly in Chrome's own built-in trace
viewer (chrome://tracing) or in specialized tools -- providing a VISUAL,
TIMELINE-based view of exactly what was happening on CPU and GPU at every
moment during the profiled code, including GAPS (idle time) that directly
reveal exactly WHERE and for HOW LONG the GPU was waiting (directly
Chapter 28's "GPU utilization" diagnostic, now visualized precisely rather
than inferred from an aggregate nvidia-smi percentage).



### Common Mistakes
- Profiling only a single forward pass in isolation, without the realistic surrounding context (data loading, the full training step including backward/optimizer) — can miss the genuine bottleneck if it lies outside the narrowly-profiled section.
- Not using `record_shapes=True` when input shape variability is a suspected contributing factor to inconsistent performance.
- Reading only the aggregate summary table without ever visually inspecting the exported trace timeline — the timeline view often reveals gaps/idle periods (directly indicating a data-pipeline or synchronization bottleneck, Chapter 28) that a purely tabular, aggregated summary can obscure.

### Interview Insight
**Q: How would you use `torch.profiler` to determine whether a slow training step is bottlenecked by the model's computation or by data loading?**
A: I'd wrap the full training step (data loading, forward pass, loss computation, backward pass, optimizer step) in the profiler with named `record_function` regions distinguishing each phase, then examine the resulting table (or, more informatively, the exported Chrome trace timeline) to see the actual time breakdown across these labeled regions. If the data-loading region consumes a disproportionate share of total time, or if the trace timeline shows clear GPU idle gaps coinciding with data-loading activity, that points to a data pipeline bottleneck (addressed via `DataLoader`'s `num_workers`, Chapter 24) rather than a model-computation bottleneck — directly distinguishing between these two very different root causes and the correspondingly different fixes each one requires, rather than guessing based on intuition alone.

### Quiz (Chapter 76)
1. Why must `ProfilerActivity.CUDA` be included to diagnose a GPU-bound bottleneck?
2. What does `record_function()` provide that profiling without it wouldn't?
3. Why is visually inspecting an exported trace timeline sometimes more revealing than reading the aggregate summary table alone?


---
# CHAPTER 77 — MEMORY OPTIMIZATION

## 1. Concept
Consolidating and extending every memory-related technique encountered across this course — batch size, gradient accumulation (Chapter 71), mixed precision (Chapter 29), and this chapter's new technique, **gradient checkpointing** — into one systematic toolkit for debugging and resolving CUDA out-of-memory errors.

## 2. The Memory Breakdown — What Actually Consumes GPU Memory


In [ ]:
print("""
GPU memory during training is consumed by FOUR main categories:

1. MODEL PARAMETERS (weights) -- fixed, proportional to model size
2. GRADIENTS -- one per trainable parameter, same size as parameters themselves
3. OPTIMIZER STATE -- Adam/AdamW's per-parameter moment estimates (Chapter 57's
   memory discussion, now generalized beyond just LLM fine-tuning)
4. ACTIVATIONS -- intermediate layer outputs saved during the forward pass,
   SPECIFICALLY needed for the backward pass's gradient computation (Chapter
   13-14) -- for a DEEP network with a LARGE batch size, activation memory
   can be the LARGEST of all four categories, and is often the LEAST
   intuitive to reason about without specifically understanding WHY it's
   needed (the backward pass's chain rule computation genuinely requires
   access to forward-pass intermediate values).
""")



GPU memory during training is consumed by FOUR main categories:

1. MODEL PARAMETERS (weights) -- fixed, proportional to model size
2. GRADIENTS -- one per trainable parameter, same size as parameters themselves
3. OPTIMIZER STATE -- Adam/AdamW's per-parameter moment estimates (Chapter 57's
   memory discussion, now generalized beyond just LLM fine-tuning)
4. ACTIVATIONS -- intermediate layer outputs saved during the forward pass,
   SPECIFICALLY needed for the backward pass's gradient computation (Chapter
   13-14) -- for a DEEP network with a LARGE batch size, activation memory
   can be the LARGEST of all four categories, and is often the LEAST
   intuitive to reason about without specifically understanding WHY it's
   needed (the backward pass's chain rule computation genuinely requires
   access to forward-pass intermediate values).



## 3. Gradient Checkpointing — Trading Compute for Activation Memory


In [ ]:
print("""
import torch.utils.checkpoint as checkpoint

class CheckpointedBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim), nn.ReLU())

    def forward(self, x):
        # checkpoint.checkpoint() runs self.block's forward WITHOUT saving its
        # intermediate activations -- then, DURING the backward pass, it
        # RE-RUNS the forward computation (just for this block) to
        # RECOMPUTE the activations it needs, on the fly, right when needed
        return checkpoint.checkpoint(self.block, x, use_reentrant=False)
""")
print("""
THE TRADEOFF, PRECISELY: gradient checkpointing SIGNIFICANTLY reduces
activation memory (since most intermediate activations are DISCARDED after
the forward pass and only RECOMPUTED when actually needed during backward),
at the cost of a GENUINE, roughly 20-30% INCREASE in total training TIME
(since the forward computation for checkpointed blocks effectively runs
TWICE -- once normally during the forward pass, once again during backward
to regenerate the discarded activations).

THIS IS A DELIBERATE, EXPLICIT TIME-FOR-MEMORY TRADE -- directly analogous
in SPIRIT to Chapter 71's gradient accumulation (which trades TIME for a
DIFFERENT kind of memory savings), but applied to ACTIVATION memory
specifically rather than batch-size-driven memory. Gradient checkpointing
is GENUINELY standard practice for training very deep/large models (common
in LLM training, Chapter 57) where activation memory alone can otherwise
make training infeasible even with otherwise-sufficient GPU memory for
weights/gradients/optimizer state.
""")



import torch.utils.checkpoint as checkpoint

class CheckpointedBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim), nn.ReLU())

    def forward(self, x):
        # checkpoint.checkpoint() runs self.block's forward WITHOUT saving its
        # intermediate activations -- then, DURING the backward pass, it
        # RE-RUNS the forward computation (just for this block) to
        # RECOMPUTE the activations it needs, on the fly, right when needed
        return checkpoint.checkpoint(self.block, x, use_reentrant=False)


THE TRADEOFF, PRECISELY: gradient checkpointing SIGNIFICANTLY reduces
activation memory (since most intermediate activations are DISCARDED after
the forward pass and only RECOMPUTED when actually needed during backward),
at the cost of a GENUINE, roughly 20-30% INCREASE in total training TIME
(since the forward computation for checkpointed blocks effectively runs


## 4. The Full Memory Optimization Toolkit, Consolidated
| Technique | What it saves | Cost |
|---|---|---|
| Smaller batch size | Activation memory (proportionally) | Can hurt GPU utilization (Chapter 28), may need accumulation to compensate |
| Gradient accumulation (Chapter 71) | Activation memory, via smaller micro-batches | No compute time savings (roughly neutral) |
| Mixed precision (Chapter 29) | Activation AND weight/gradient memory (roughly halved) | Minimal — often a "free" win on appropriate hardware |
| Gradient checkpointing (this chapter) | Activation memory specifically, substantially | ~20-30% more training time |
| Quantization (Chapter 59) | Weight memory specifically (inference only, typically) | Some accuracy risk, requires care |

### Common Mistakes
- Reaching for gradient checkpointing as a first resort, before trying simpler/cheaper options (smaller batch size, mixed precision, gradient accumulation) — checkpointing's real time cost means it's best reserved for when those other, cheaper techniques genuinely aren't sufficient.
- Not understanding that gradient checkpointing's benefit is specifically for **activation** memory — it does nothing to reduce weight, gradient, or optimizer-state memory, so it won't help if those are the actual dominant memory consumer for your specific model/configuration.
- Combining every memory-saving technique indiscriminately without measuring whether each one is actually needed/helping — each has its own cost (time, complexity, or accuracy), and a measured, incremental approach (Scikit-learn course's Chapter 61 profiling discipline) is more disciplined than applying everything at once.

### Interview Insight
**Q: You're training a very deep model and hit CUDA OOM. Walk through your systematic approach to resolving it, in order of what you'd try first.**
A: First, confirm `torch.inference_mode()` is correctly used anywhere evaluation/validation happens (Chapter 15) — a surprisingly common, easy-to-overlook source of unnecessary memory usage. Then, try the cheapest interventions first: reduce batch size (while using gradient accumulation, Chapter 71, to maintain the same effective batch size if the smaller true batch would hurt training quality), and enable mixed precision training (Chapter 29), which often provides substantial memory savings nearly for free on appropriate hardware. If OOM persists and the model is genuinely very deep (meaning activation memory specifically is likely the dominant consumer), gradient checkpointing (this chapter) directly targets that specific memory category, at the cost of roughly 20-30% more training time — a worthwhile, deliberate trade when it's genuinely what's needed to make training feasible at all. I'd measure memory usage at each step (via `torch.cuda.memory_allocated()`, Chapter 65) to confirm each intervention is actually helping, rather than applying every technique reflexively.

### Quiz (Chapter 77)
1. List the four categories of GPU memory consumption during training.
2. What specifically does gradient checkpointing trade, and what's the approximate cost?
3. Why would gradient checkpointing NOT help if your memory problem is actually dominated by optimizer state, not activations?
4. In what order would you try the memory optimization toolkit's techniques, and why that order?


---
# CHAPTER 78 — GRADIENT CLIPPING

## 1. Concept
Gradient clipping directly, mechanically prevents the exploding-gradient failure mode (Chapters 14, 32-33) by capping the **magnitude** of gradients before the optimizer applies its update — a simple, standard, near-universal technique, especially for RNN/LSTM (Chapter 44-45) and Transformer (Chapter 50) training.

## 2. `torch.nn.utils.clip_grad_norm_`


In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(10, 5)
x = torch.randn(4, 10)
y = torch.randn(4, 5)

logits = model(x)
loss = ((logits - y) ** 2).sum() * 1000   # DELIBERATELY inflated loss, to simulate exploding gradients
loss.backward()

# Check gradient norm BEFORE clipping
total_norm_before = torch.norm(torch.stack([p.grad.norm() for p in model.parameters()]))
print(f"Total gradient norm BEFORE clipping: {total_norm_before.item():.2f}")

# clip_grad_norm_ rescales ALL gradients TOGETHER (preserving their RELATIVE
# directions/proportions) so the TOTAL norm does not exceed max_norm
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

total_norm_after = torch.norm(torch.stack([p.grad.norm() for p in model.parameters()]))
print(f"Total gradient norm AFTER clipping: {total_norm_after.item():.2f}")
print("\nNote: clipping happens AFTER loss.backward() but BEFORE optimizer.step() --")
print("exactly where it needs to sit in the canonical training step sequence.")


Total gradient norm BEFORE clipping: 22757.98
Total gradient norm AFTER clipping: 1.00

Note: clipping happens AFTER loss.backward() but BEFORE optimizer.step() --
exactly where it needs to sit in the canonical training step sequence.


### Line-by-Line Explanation — Clipping by NORM, Not by Value
- `clip_grad_norm_` computes the **total** norm across **all** parameters' gradients combined (treating them conceptually as one giant concatenated vector), and if that total exceeds `max_norm`, rescales **every** parameter's gradient by the **same** scaling factor — critically, this **preserves the relative direction/proportions** between different parameters' gradients, just uniformly shrinking the overall step size. This is meaningfully different from (and generally preferred over) `clip_grad_value_`, which clips each individual gradient *value* independently to a fixed range — value-clipping can distort the gradient's overall *direction* (since different components might get clipped by different relative amounts), while norm-clipping preserves direction and only affects overall magnitude.
- The trailing underscore in `clip_grad_norm_` follows PyTorch's standard **in-place operation** naming convention (Chapter 9's lesson, now encountered in a new context) — it modifies `.grad` attributes directly, in place, rather than returning a new value you'd need to reassign.
- **Where it sits in the training step**: after `loss.backward()` (gradients must already be computed) but before `optimizer.step()` (the clipped, not raw, gradients should be what the optimizer actually applies) — a specific, important ordering requirement.

## 3. The Canonical Training Step With Gradient Clipping


In [ ]:
def train_step_with_clipping(model, X_batch, y_batch, criterion, optimizer, max_norm=1.0):
    optimizer.zero_grad()
    logits = model(X_batch)
    loss = criterion(logits, y_batch)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)   # THE ONE ADDITIONAL LINE

    optimizer.step()
    return loss.item()

print("Gradient clipping requires exactly ONE additional line in the canonical")
print("training step (Chapter 20) -- inserted between backward() and step() --")
print("genuinely one of the simplest, highest-value additions for training stability,")
print("especially for RNN/LSTM/Transformer architectures (Chapters 44-51).")


Gradient clipping requires exactly ONE additional line in the canonical
training step (Chapter 20) -- inserted between backward() and step() --
genuinely one of the simplest, highest-value additions for training stability,
especially for RNN/LSTM/Transformer architectures (Chapters 44-51).


### Choosing `max_norm` — a Genuine, Task-Dependent Hyperparameter
There's no universal "correct" `max_norm` value — common starting points range from `0.5` to `5.0` depending on architecture and task, and like any hyperparameter, the right value is best determined empirically (checking whether training stability improves, via the gradient-norm and loss-curve diagnostics from Chapter 32) rather than assumed from a single rule of thumb.

### Common Mistakes
- Clipping gradients **before** `loss.backward()` — gradients don't exist yet at that point; clipping must happen after backward, before the optimizer step.
- Using `clip_grad_value_` (per-element clipping) when `clip_grad_norm_` (preserving gradient direction) would be more appropriate — value-based clipping can distort the gradient's overall direction in ways norm-based clipping avoids.
- Setting `max_norm` without ever checking whether it's actually binding (actively clipping) during training — if gradients rarely or never exceed the threshold, clipping provides no real protective benefit, and if it's binding on nearly every step, the threshold may be set too aggressively low, itself hurting training.
- Treating gradient clipping as a substitute for addressing a genuinely broken learning rate or data issue — clipping treats a *symptom* (excessively large gradient steps) and is often used *alongside* proper learning rate tuning, not as a replacement for it.

### Interview Insight
**Q: Why is `clip_grad_norm_` (clipping the total gradient norm) generally preferred over `clip_grad_value_` (clipping each gradient element independently)?**
A: `clip_grad_value_` clips each individual gradient value to a fixed range independently of every other value — if different parameters' gradients happen to have very different magnitudes, this can clip some components proportionally much more aggressively than others, distorting the gradient vector's overall *direction*, not just its magnitude — potentially sending the optimizer in a meaningfully different direction than the true (unclipped) gradient would have indicated. `clip_grad_norm_` instead computes one total norm across all parameters combined and, if it exceeds the threshold, rescales every gradient by the exact same factor — this uniform rescaling preserves the gradient's overall direction/proportions entirely, only capping its total magnitude, which is generally the more principled, less distorting way to prevent exploding-gradient instability while still taking a genuinely representative (just appropriately-sized) step in the correct direction.

### Quiz (Chapter 78)
1. Where exactly in the training step must gradient clipping be inserted, and why that specific position?
2. Why does `clip_grad_norm_` preserve gradient direction while `clip_grad_value_` can distort it?
3. How would you determine whether your chosen `max_norm` value is actually providing a meaningful protective effect during training?

### Chapter 78 Summary
- Gradient clipping is a single-line, near-universal addition to the standard training step, inserted between `backward()` and `step()`.
- Norm-based clipping (`clip_grad_norm_`) is generally preferred over value-based clipping for preserving the gradient's overall direction while capping its magnitude.
- `max_norm` is a genuine hyperparameter requiring empirical tuning, not a fixed universal constant.

---

## END OF NOTEBOOK 13 — What You Should Now Be Able To Do
- Save and resume training correctly with a complete checkpoint (model, optimizer, scheduler state), including correct cross-device loading via `map_location`.
- Build multi-input, multi-output architectures with appropriately weighted combined losses for joint multi-task training.
- Implement a custom `torch.autograd.Function`, understanding precisely how `ctx.save_for_backward` and `backward()`'s `grad_output` connect directly to the chain rule's mechanics.
- Apply `torch.compile` correctly, understanding the first-call compilation cost and benchmarking its steady-state speedup fairly.
- Use `torch.profiler` to diagnose genuine bottlenecks with operation-level precision, distinguishing data-pipeline from model-computation costs.
- Apply the complete memory optimization toolkit (batch size, accumulation, mixed precision, gradient checkpointing, quantization) systematically, understanding each technique's specific tradeoff.
- Apply gradient clipping correctly as a standard training-stability technique, understanding why norm-based clipping is generally preferred.

## Before Moving to Notebook 14 (Chapters 79–85: Large-Scale Training, Production Project Structure, Testing PyTorch Code, End-to-End Computer Vision Project, End-to-End NLP Project, End-to-End LLM Project, End-to-End Recommendation Project)
 